# Bert. Pure pytorch version

## Load data

In [1]:
!curl -L -o ~/invoices-ocr.zip https://www.kaggle.com/api/v1/datasets/download/osamahosamabdellatif/high-quality-invoice-images-for-ocr
!unzip ~/invoices-ocr.zip -d /content/invoices

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1165M  100 1165M    0     0  14.3M      0  0:01:21  0:01:21 --:--:-- 14.9M
Archive:  /root/invoices-ocr.zip
  inflating: /content/invoices/batch_1/batch_1/batch1_1.csv  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0001.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0002.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0003.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0004.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0005.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0006.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0007.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0008.jpg

In [1]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    scikit-learn \
    torchmetrics \
    httpx
!pip install -q --use-pep517 seqeval
!pip install rapidfuzz

In [2]:
%pip uninstall -y ipywidgets
%pip install ipywidgets==7.7.2

Found existing installation: ipywidgets 7.7.1
Uninstalling ipywidgets-7.7.1:
  Successfully uninstalled ipywidgets-7.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 169.2 MB/s  0:00:00
  Attempting uninstall: jupyterlab-widgets
    Found existing installation: jupyterlab_widgets 3.0.17
    Uninstalling jupyterlab_widgets-3.0.17:
      Successfully uninstalled jupyterlab_widgets-3.0.17
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [jedi]


In [19]:
import json
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from pathlib import Path

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch
from torch import nn

from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    set_seed
)
from pathlib import Path
from zipfile import ZipFile

import httpx

from tqdm import tqdm

In [ ]:
KAGGLE_INVOICES_URL = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "osamahosamabdellatif/high-quality-invoice-images-for-ocr"
)

@dataclass
class InvoiceDatasetDownloader:
    destination: str | Path = "/content/invoices"
    archive_path: str | Path = "/content/invoices-ocr.zip"

    def __post_init__(self):
        self.destination = Path(self.destination).expanduser()
        self.archive_path = Path(self.archive_path).expanduser()

    def download(self) -> Path:
        """Reuse an existing archive or download it with progress reporting."""
    
        if self.archive_path.is_file():
            return self.archive_path

        self.archive_path.parent.mkdir(parents=True, exist_ok=True)
        temporary_path = self.archive_path.with_suffix(".zip.part")

        try:
            with httpx.stream(
                "GET",
                KAGGLE_INVOICES_URL,
                follow_redirects=True,
                timeout=httpx.Timeout(120.0, connect=30.0),
            ) as response:
                response.raise_for_status()

                content_length = response.headers.get("Content-Length")
                total_bytes = int(content_length) if content_length else None

                with (
                    temporary_path.open("wb") as file,
                    tqdm(
                        total=total_bytes,
                        desc="Downloading invoices",
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                    ) as progress,
                ):
                    for chunk in response.iter_bytes(
                        chunk_size=1024 * 1024
                    ):
                        file.write(chunk)
                        progress.update(len(chunk))

            temporary_path.replace(self.archive_path)
        finally:
            temporary_path.unlink(missing_ok=True)

        return self.archive_path

    def extract(self) -> Path:
        """Extract the downloaded ZIP, overwriting files with matching names."""
        self.destination.mkdir(parents=True, exist_ok=True)

        with ZipFile(self.archive_path) as archive:
            archive.extractall(self.destination)

        return self.destination

    def download_and_extract(self) -> Path:
        self.download()
        return self.extract()

In [18]:
downloader = InvoiceDatasetDownloader(
    destination="/content/invoices_test", 
    archive_path="/content/invoices-ocr-test.zip",
)
invoices_path = downloader.download_and_extract()
invoices_path

PosixPath('/content/invoices_test')

In [ ]:
@dataclass
class InvoiceDatasetLoader:
    invoices_path: str | Path

    def __post_init__(self):
        self.invoices_path = Path(self.invoices_path).expanduser()

    def load(self) -> pd.DataFrame:
        dfs = []
        for csv_path in sorted(self.invoices_path.rglob("*.csv")):
            image_dir = csv_path.parent / csv_path.stem

            if not image_dir.is_dir():
                print(f"Skipping: image directory not found for {csv_path}")
                continue

            df_part = pd.read_csv(csv_path)

            if "File Name" not in df_part.columns:
                print(f"Skipping: 'File Name' column not found in {csv_path}")
                continue

            # Keep provenance because folder names repeat in different batches.
            df_part["subdir"] = csv_path.stem
            df_part["source_group"] = str(csv_path.parent.relative_to(self.invoices_path))
            df_part["csv_path"] = str(csv_path)

            df_part["path"] = df_part["File Name"].map(
                lambda filename: str(image_dir / filename)
            )

            dfs.append(df_part)

        if not dfs:
            raise RuntimeError("No valid CSV files were found")

        df = pd.concat(dfs, ignore_index=True)

        # Normalize column names.
        df.columns = (
            df.columns
            .str.strip()
            .str.replace(r"(?<=[a-z0-9])(?=[A-Z])", "_", regex=True)
            .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
            .str.strip("_")
            .str.lower()
        )
        return df


In [21]:
df = InvoiceDatasetLoader(invoices_path).load()
df

,file_name,json_data,ocred_text,subdir,source_group,csv_path,path
0,batch1-0494.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Clark...",Invoice no: 84652373 Date of issue: 02/23/2021...,batch1_1,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
1,batch1-0489.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Willi...",Invoice no: 37451664 Date of issue: 06/11/2020...,batch1_1,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
2,batch1-0499.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Heste...",Invoice no: 40108666 Date of issue: 02/07/2020...,batch1_1,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
3,batch1-0497.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Olson...",Invoice no: 73285932 Date of issue: 07/25/2017...,batch1_1,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
4,batch1-0081.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Wilso...",Invoice no: 15288019 Date of issue: 09/07/2014...,batch1_1,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
...,...,...,...,...,...,...,...
1409,batch1-1359.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Dawso...",Invoice no: 87519797 Date of issue: 05/13/2013...,batch1_3,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
1410,batch1-1391.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Brock...",Invoice no: 94223548 Date of issue: 11/19/2012...,batch1_3,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
1411,batch1-1375.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Flynn...",Invoice no: 59612541 Date of issue: 08/24/2016...,batch1_3,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...
1412,batch1-1412.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Green...",Invoice no: 34630909 Date of issue: 09/14/2011...,batch1_3,batch_1/batch_1,/content/invoices_test/batch_1/batch_1/batch1_...,/content/invoices_test/batch_1/batch_1/batch1_...


In [86]:
SEED = 42
MODEL_NAME = "google-bert/bert-base-uncased"
MAX_LENGTH=512
BATCH_SIZE=16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


In [6]:
invoices_path = Path("/content/invoices/")
dfs = []
for csv_path in sorted(invoices_path.rglob("*.csv")):
    image_dir = csv_path.parent / csv_path.stem

    if not image_dir.is_dir():
        print(f"Skipping: image directory not found for {csv_path}")
        continue

    df_part = pd.read_csv(csv_path)

    if "File Name" not in df_part.columns:
        print(f"Skipping: 'File Name' column not found in {csv_path}")
        continue

    # Keep provenance because folder names repeat in different batches.
    df_part["subdir"] = csv_path.stem
    df_part["source_group"] = str(csv_path.parent.relative_to(invoices_path))
    df_part["csv_path"] = str(csv_path)

    df_part["path"] = df_part["File Name"].map(
        lambda filename: str(image_dir / filename)
    )

    dfs.append(df_part)

if not dfs:
    raise RuntimeError("No valid CSV files were found")

df = pd.concat(dfs, ignore_index=True)

# Normalize column names.
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"(?<=[a-z0-9])(?=[A-Z])", "_", regex=True)
    .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)


## Tokenizer

### Goal

```json
{
    "text": "Supplier: Acme GmbH. Total: 125 EUR.",
    "entities": [
        {"start": 10, "end": 19, "label": "SUPPLIER"},
        {"start": 28, "end": 31, "label": "TOTAL"},
        {"start": 32, "end": 35, "label": "CURRENCY"},
    ],
}
```

### Investigate data

In [7]:
row = df.iloc[0]
ocr_text = row.ocred_text
json_data = json.loads(row.json_data)

In [8]:
ocr_text

'Invoice no: 84652373 Date of issue: 02/23/2021 Seller: Client: Nguyen-Roach Clark-Foster 247 David Highway 77477 Cliff Apt. 853 Lake John, WV 84178 Washingtonbury, MS 78346 Tax Id: 991-72-5826 Tax Id: 937-70-8530 IBAN: GB91/YXO05542456978150 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Stemware Rack Display Kitchen 1,00 each 42,32 42,32 10% 46,55 Wine Glass Holder Bottle Carbon Steel Free Punch 2 VTG (4) 7 Ounce Since 1852 1,00 each 14,00 14,00 10% 15,40 Milk Bottle Wine Carafe Juice Glass with Cork Lids 3 Vintage Crystal Red Wine 1,00 each 35,45 35,45 10% 39,00 Glasses NOS West Germany 1983 6 10 ounce elegant stems 3 Ikea Stainless Steel 4-bottle 4,00 each 25,00 100,00 10% 110,00 Wine Rack 300.557.60 great condition gift it! 5 Lolita "Wine Bouquet" Hand 1,00 each 20,00 20,00 10% 22,00 Painted and Decorated Wine Glass NIB SUMMARY VAT [%] Net worth VAT Gross worth 10% 211,77 21,18 232,95 Total $ 211,77 $ 21,18 $ 232,95 Troy'

In [9]:
json_data

{'invoice': {'client_name': 'Clark-Foster',
  'client_address': '77477 Troy Cliff Apt. 853\nWashingtonbury, MS 78346',
  'seller_name': 'Nguyen-Roach',
  'seller_address': '247 David Highway\nLake John, WV 84178',
  'invoice_number': '84652373',
  'invoice_date': '02/23/2021',
  'due_date': ''},
 'items': [{'description': 'Stemware Rack Display Kitchen\nWine Glass Holder Bottle\nCarbon Steel Free Punch',
   'quantity': '1.00',
   'total_price': '46.55'},
  {'description': 'VTG (4) 7 Ounce Since 1852\nMilk Bottle Wine Carafe Juice\nGlass with Cork Lids',
   'quantity': '1.00',
   'total_price': '15.40'},
  {'description': 'Vintage Crystal Red Wine\nGlasses NOS West Germany\n1983 6 10 ounce elegant stems',
   'quantity': '1.00',
   'total_price': '39.00'},
  {'description': '3 Ikea Stainless Steel 4-bottle\nWine Rack 300.557.60 - great\ncondition gift it!',
   'quantity': '4.00',
   'total_price': '110.00'},
  {'description': 'Lolita "Wine Bouquet" Hand\nPainted and Decorated Wine\nGlass

In [10]:
def flatten_field_paths(
    data: dict,
    prefix: str = "",
) -> list[str]:
    """Return dotted paths for non-empty string values in a nested JSON object."""
    paths = []

    for key, value in data.items():
        path = f"{prefix}.{key}" if prefix else key

        if isinstance(value, dict):
            paths.extend(flatten_field_paths(value, path))
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    paths.extend(flatten_field_paths(item, path))
        elif isinstance(value, str) and value.strip():
            paths.append(path)

    return paths

In [11]:
set(flatten_field_paths(json_data))

{'invoice.client_address',
 'invoice.client_name',
 'invoice.invoice_date',
 'invoice.invoice_number',
 'invoice.seller_address',
 'invoice.seller_name',
 'items.description',
 'items.quantity',
 'items.total_price',
 'subtotal.tax',
 'subtotal.total'}

In [12]:
from collections import Counter

field_document_counts = Counter()

for json_data in df["json_data"]:
    target = json.loads(json_data) if isinstance(json_data, str) else json_data

    # A set ensures a field is counted at most once per document.
    field_document_counts.update(set(flatten_field_paths(target)))

field_coverage = (
    pd.DataFrame(
        field_document_counts.items(),
        columns=["field", "documents_with_field"],
    )
    .sort_values("documents_with_field", ascending=False)
    .reset_index(drop=True)
)

field_coverage["coverage"] = (
    field_coverage["documents_with_field"] / len(df)
)

In [13]:
field_coverage['field'].to_list()

['invoice.invoice_number',
 'items.quantity',
 'invoice.seller_address',
 'invoice.invoice_date',
 'subtotal.tax',
 'invoice.client_address',
 'subtotal.total',
 'items.total_price',
 'invoice.seller_name',
 'items.description',
 'invoice.client_name',
 'payment_instructions.account_number',
 'payment_instructions.bank_name']

### Build BIO token labels for BERT

In [30]:
FIELD_TO_ENTITY = {
    "invoice.client_address": "CLIENT_ADDRESS",
    "invoice.client_name": "CLIENT_NAME",
    "invoice.seller_address": "SELLER_ADDRESS",
    "invoice.seller_name": "SELLER_NAME",
    "invoice.invoice_number": "INVOICE_NUMBER",
    "invoice.invoice_date": "INVOICE_DATE",
    "payment_instructions.account_number": "ACCOUNT_NUMBER",
    "subtotal.tax": "TAX",
    "subtotal.total": "TOTAL",
}

ENTITY_TO_FIELD = {
    entity: field
    for field, entity in FIELD_TO_ENTITY.items()
}

LABELS = ["O"] + [
    f"{prefix}-{entity}"
    for entity in FIELD_TO_ENTITY.values()
    for prefix in ("B", "I")
]

label2id = {
    label: index
    for index, label in enumerate(LABELS)
}

id2label = dict(enumerate(LABELS))

LABELS

['O',
 'B-CLIENT_ADDRESS',
 'I-CLIENT_ADDRESS',
 'B-CLIENT_NAME',
 'I-CLIENT_NAME',
 'B-SELLER_ADDRESS',
 'I-SELLER_ADDRESS',
 'B-SELLER_NAME',
 'I-SELLER_NAME',
 'B-INVOICE_NUMBER',
 'I-INVOICE_NUMBER',
 'B-INVOICE_DATE',
 'I-INVOICE_DATE',
 'B-ACCOUNT_NUMBER',
 'I-ACCOUNT_NUMBER',
 'B-TAX',
 'I-TAX',
 'B-TOTAL',
 'I-TOTAL']

### Find spans

Write function that will find spans in ocr text by given json_data

In [15]:
row = df.iloc[0]
ocr_text = row.ocred_text
json_data = json.loads(row.json_data)

In [16]:
json_data

{'invoice': {'client_name': 'Clark-Foster',
  'client_address': '77477 Troy Cliff Apt. 853\nWashingtonbury, MS 78346',
  'seller_name': 'Nguyen-Roach',
  'seller_address': '247 David Highway\nLake John, WV 84178',
  'invoice_number': '84652373',
  'invoice_date': '02/23/2021',
  'due_date': ''},
 'items': [{'description': 'Stemware Rack Display Kitchen\nWine Glass Holder Bottle\nCarbon Steel Free Punch',
   'quantity': '1.00',
   'total_price': '46.55'},
  {'description': 'VTG (4) 7 Ounce Since 1852\nMilk Bottle Wine Carafe Juice\nGlass with Cork Lids',
   'quantity': '1.00',
   'total_price': '15.40'},
  {'description': 'Vintage Crystal Red Wine\nGlasses NOS West Germany\n1983 6 10 ounce elegant stems',
   'quantity': '1.00',
   'total_price': '39.00'},
  {'description': '3 Ikea Stainless Steel 4-bottle\nWine Rack 300.557.60 - great\ncondition gift it!',
   'quantity': '4.00',
   'total_price': '110.00'},
  {'description': 'Lolita "Wine Bouquet" Hand\nPainted and Decorated Wine\nGlass

In [34]:
ocr_text

'Invoice no: 84652373 Date of issue: 02/23/2021 Seller: Client: Nguyen-Roach Clark-Foster 247 David Highway 77477 Cliff Apt. 853 Lake John, WV 84178 Washingtonbury, MS 78346 Tax Id: 991-72-5826 Tax Id: 937-70-8530 IBAN: GB91/YXO05542456978150 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Stemware Rack Display Kitchen 1,00 each 42,32 42,32 10% 46,55 Wine Glass Holder Bottle Carbon Steel Free Punch 2 VTG (4) 7 Ounce Since 1852 1,00 each 14,00 14,00 10% 15,40 Milk Bottle Wine Carafe Juice Glass with Cork Lids 3 Vintage Crystal Red Wine 1,00 each 35,45 35,45 10% 39,00 Glasses NOS West Germany 1983 6 10 ounce elegant stems 3 Ikea Stainless Steel 4-bottle 4,00 each 25,00 100,00 10% 110,00 Wine Rack 300.557.60 great condition gift it! 5 Lolita "Wine Bouquet" Hand 1,00 each 20,00 20,00 10% 22,00 Painted and Decorated Wine Glass NIB SUMMARY VAT [%] Net worth VAT Gross worth 10% 211,77 21,18 232,95 Total $ 211,77 $ 21,18 $ 232,95 Troy'

In [ ]:
import re
import unicodedata
import pandas as pd
import json
from decimal import Decimal, InvalidOperation
from rapidfuzz import fuzz
from difflib import SequenceMatcher
import contextlib
import io

ACCOUNT_NUMBER_MIN_SCORE = 80

NUMERIC_FIELD_NAMES = {
    "quantity",
    "total_price",
    "tax",
    "discount",
    "total",
}

def is_numeric_field(path: str) -> bool:
    field_name = path.rsplit(".", 1)[-1].split("[", 1)[0]
    return field_name in NUMERIC_FIELD_NAMES

def number_pattern(value: str) -> re.Pattern:
    """Build a regex for a numeric field, allowing decimal/grouping variants.

    The rightmost dot or comma is treated as the decimal separator when the
    value has digits after it. Group separators and spaces between integer
    digits are accepted, while the decimal boundary and fractional digits are
    kept significant (so 13.99 cannot match 139,9).
    """
    is_percentage = unicodedata.normalize("NFKC", str(value)).strip().endswith("%")
    amount = normalize_number(value, percentage=is_percentage)
    if amount is None:
        raise ValueError(f"Invalid or ambiguous invoice amount: {value!r}")
    number_text = format(amount, "f")

    sign = r"[-+]?"
    if number_text.startswith(("-", "+")):
        sign = re.escape(number_text[0])
        number_text = number_text[1:].strip()

    decimal_pos = max(number_text.rfind("."), number_text.rfind(","))
    if decimal_pos >= 0 and number_text[decimal_pos + 1 :].isdigit():
        integer_part = number_text[:decimal_pos]
        fractional_part = number_text[decimal_pos + 1 :]
    else:
        integer_part = number_text
        fractional_part = ""

    integer_digits = re.findall(r"\d", integer_part)
    if not integer_digits:
        raise ValueError(f"Numeric field has no integer digits: {value!r}")

    # Permit OCR differences in thousands separators, e.g. 1 234,20 / 1234.20.
    integer_pattern = r"[\s.,'’]*".join(map(re.escape, integer_digits))
    number_pattern_text = sign + integer_pattern

    if fractional_part:
        number_pattern_text += r"[.,]" + re.escape(fractional_part)

    if is_percentage:
        number_pattern_text += r"\s*%"

    return re.compile(rf"(?<!\d){number_pattern_text}(?!\d)")

def normalize_with_offsets(text: str) -> tuple[str, list[int]]:
    """Normalize text for matching while mapping each output character to OCR.

    Punctuation and whitespace form one separator, preserving word boundaries.
    Unicode expansions map back to the original character that produced them.
    """
    normalized_chars = []
    original_offsets = []

    for index, char in enumerate(text):
        char = unicodedata.normalize("NFKC", char).casefold()
        for normalized_char in char:
            if normalized_char.isalnum():
                normalized_chars.append(normalized_char)
                original_offsets.append(index)
            elif normalized_chars and normalized_chars[-1] != " ":
                normalized_chars.append(" ")
                original_offsets.append(index)

    if normalized_chars and normalized_chars[-1] == " ":
        normalized_chars.pop()
        original_offsets.pop()

    return "".join(normalized_chars), original_offsets


def find_normalized_span(text: str, value: str, label: str) -> dict | None:
    normalized_text, offsets = normalize_with_offsets(text)
    normalized_value, _ = normalize_with_offsets(value)

    if not normalized_value:
        return None

    match = fuzz.partial_ratio_alignment(
        normalized_value,
        normalized_text
    )
    if not match:
        return

    start = offsets[match.dest_start]
    end = offsets[match.dest_end - 1] + 1
    matched_ocr_text = text[start:end]

    # Reject a mapped range if it doesn't normalize back to the requested value.
    checked_text, _ = normalize_with_offsets(matched_ocr_text)
    if checked_text != normalized_value:
        return None

    return {
        "start": start,
        "end": end,
        "label": label,
        "text": matched_ocr_text,
    }

def find_fuzzy_description_candidate(
    text: str,
    fragment: str,
    item_start: int,
    item_end: int,
    min_score: float = 85,
) -> dict | None:
    """Suggest an OCR span for one unmatched description fragment."""
    item_text = text[item_start:item_end]

    normalized_text, offsets = normalize_with_offsets(item_text)
    normalized_fragment, _ = normalize_with_offsets(fragment)

    if not normalized_text or not normalized_fragment:
        return None

    match = fuzz.partial_ratio_alignment(
        normalized_fragment,
        normalized_text,
        score_cutoff=min_score,
    )
    if match is None:
        return None

    start = item_start + offsets[match.dest_start]
    end = item_start + offsets[match.dest_end - 1] + 1

    return {
        "start": start,
        "end": end,
        "text": text[start:end],
        "expected": fragment,
        "score": match.score,
    }

def find_description_spans(
    text: str,
    description: str,
    label: str
) -> tuple[list[dict], list[str]]:
    """Find description lines within an item region.

    Return matched OCR spans and lines that could not be matched uniquely.
    """
    norm_text, offsets = normalize_with_offsets(text)
    spans = []
    unmatched = []

    for fragment in description.splitlines():
        norm_fragment, _ = normalize_with_offsets(fragment)
        if not norm_fragment:
            continue
        match = fuzz.partial_ratio_alignment(
            norm_fragment,
            norm_text
        )
        if match is None or match.dest_start == match.dest_end:
            unmatched.append(fragment)
            continue

        start = offsets[match.dest_start]
        end = offsets[match.dest_end-1] + 1
        spans.append({
            "start": start,
            "end": end,
            "label": label,
            "text": text[start:end+1].strip(),
            "score": match.score,
        })

    return spans, unmatched

def find_name_span(text: str, expected: str, label: str) -> dict | None:
    """Accept exact names or ordered partial names with the first name word.

    Partial candidates must contain at least two expected words and no inserted
    words inside their boundaries. Other fuzzy candidates require review.
    This checks text alignment; it cannot establish seller/client context.
    """
    exact = find_normalized_span(text, expected, label)
    if exact is not None:
        return {**exact, "needs_review": False, "match_status": "exact"}

    normalized_text, offsets = normalize_with_offsets(text)
    normalized_expected = normalize(expected)
    if not normalized_text or not normalized_expected:
        return None
    match = fuzz.partial_ratio_alignment(normalized_expected, normalized_text)
    if match is None or match.dest_start == match.dest_end:
        return None

    candidate_tokens = list(re.finditer(r"\S+", normalized_text[match.dest_start:match.dest_end]))
    expected_words = normalized_expected.split()
    blocks = [block for block in SequenceMatcher(
        None, expected_words, [token.group() for token in candidate_tokens],
        autojunk=False,
    ).get_matching_blocks() if block.size]

    start = offsets[match.dest_start]
    end = offsets[match.dest_end - 1] + 1
    partial = False
    if blocks:
        first, last = blocks[0], blocks[-1]
        matched_words = sum(block.size for block in blocks)
        # Ignore unrelated words at the edges, but reject interior insertions.
        if first.a == 0 and matched_words >= 2 and last.b + last.size - first.b == matched_words:
            left = match.dest_start + candidate_tokens[first.b].start()
            right = match.dest_start + candidate_tokens[last.b + last.size - 1].end()
            start, end = offsets[left], offsets[right - 1] + 1
            partial = True

    observed = text[start:end]
    return {"start": start, "end": end, "label": label, "text": observed,
            "needs_review": not partial,
            "match_status": "partial" if partial else "candidate",
            "expected": expected,
            "score": fuzz.ratio(normalized_expected, normalize(observed))}


def find_summary_spans(
    text: str,
    json_data: dict,
    lookahead: int = 200,
) -> dict[str, dict]:
    """Align JSON subtotal values to complete tokens in the summary.

    Search monetary amounts after the final Total label and percentages
    between the preceding SUMMARY label and Total. Accept only unique
    numerically equal candidates.

    Args:
        text: the original OCR string
        json_data: The expected annotations
        lookahead: how many characters after Total to search
    Returns:
        Vlaue maps field paths to span dictionaries
    """

    # Find all occurrences of the word Total
    anchors = list(re.finditer(
        r"\bTotal\b",
        text,
        flags=re.IGNORECASE,
    ))

    if not anchors:
        return {}

    # Select the last occurrence
    # This reduces the chance of selecting an earlier occurrence in a descriptio
    anchor = anchors[-1]
    # Set the search boundary to at most 200 characters after the anchor.
    # min() prevents the boundary from exceeding the document length
    region_end = min(len(text), anchor.end() + lookahead)
    
    # Define a complete monetary amount
    # The regex:
    #   * (?<![\w.,]) - The preceding character must not be a word character, comma or dot
    #   * [+-]? - Permit an optional sign
    #   * \d{1,3}(?:[ \u00a0.'’]\d{3})+ - Permit grouped integer digits, such as `2 731``
    #   * `|`` between the alternatives - Choose either grouped digits or the ungrouped alternative
    #   * \d+ - Permit ungrouped digits, such as `2731`
    #   * [,.]\d{2} - Require a decimal separator followed by exactly two digits
    #   * (?![\d.,]) - Don’t stop immediately before another digit, comma or dot
    #   * \u00a0 - represents a nonbreaking space, which OCR can use as a thousands separator.
    # Examples this pattern can capture:
    #   * 273,18
    #   * 273.18
    #   * 3 004,96
    #   * 3004.96
    money_pattern = re.compile(
        r"(?<![\w.,])"
        r"[+-]?"
        r"(?:\d{1,3}(?:[ \u00a0.'’]\d{3})+|\d+)"
        r"[,.]\d{2}"
        r"(?![\d.,])"
    )

    # Extract amounts after `Total`
    # These matches already contain offsets into the original full text.
    # Search the original text, beginning immediately after Total and stopping at region_end.
    # Example:
    # ["2 731,78", "273,18", "3 004,96"]
    amounts = list(money_pattern.finditer(
        text,
        anchor.end(),
        region_end,
    ))

    # Require the expected three-amount structure.
    # If OCR loses an amount—or the 200-character region contains an 
    # extra amount—the function returns {}
    # It does not partially process that row
    if len(amounts) != 3:
        return {}
    
    # Find the preceding `SUMMARY` section
    # Search only the text before the selected Total for `SUMMARY`.
    # This prepares a second search area for percentage targets such as "10%"
    summary_anchors = list(re.finditer(
        r"\bSUMMARY\b",
        text[:anchor.start()],
        flags=re.IGNORECASE,
    ))

    # Start with no percentage candidates.
    percentages = []

    # Only search percentages if a preceding summary anchor was found.
    if summary_anchors:
        # This matches values such as:
        #   * 10%
        #   * 10 %
        #   * 7.5%
        #   * 7,5%
        # The decimal portion and whitespace before % are optional.
        percentage_pattern = re.compile(
            r"(?<![\w.,])\d+(?:[,.]\d+)?\s*%"
        )

        # Search between the last preceding `SUMMARY` and the selected `Total`.
        percentages = list(percentage_pattern.finditer(
            text,
            summary_anchors[-1].end(),
            anchor.start(),
        ))

    result = {}

    # Process the two expected fields
    # Run the following logic once for tax, then once for total.
    for field in ("tax", "total"):
        # Read the expected field safely.
        # For the example:
        #   First iteration
        #       field = "tax"
        #       expected = "273.18"
        #   Second iteration
        #       field = "total"
        #       expected = "3004.96"
        # If subtotal or the field is absent, use "".
        expected = str(
            json_data.get("subtotal", {}).get(field, "")
        ).strip()

        if not expected:
            continue
        
        # Determine whether this particular target is a percentage
        is_percentage = expected.endswith("%")

        # Convert the expected value into a Decimal:
        # Examples:
        #   "273.18" => Decimal("273.18")
        #   "3004.96" => Decimal("3004.96")
        #   "10%" => Decimal("10")  # when percentage=True
        expected_number = normalize_number(
            expected,
            percentage=is_percentage,
        )

        # Skip a value that the number parser cannot interpret safely.
        if expected_number is None:
            continue
        
        # Choose monetary or percentage candidates
        #   For "273.18", use the three monetary amounts.
        #   For "10%", use percentage matches from the preceding summary section.
        available = percentages if is_percentage else amounts

        # candidate.group() returns the actual matched OCR substring.
        # For tax = "273.18", the comparisons are:
        #   Decimal("2731.78") == Decimal("273.18")  # False
        #   Decimal("273.18")  == Decimal("273.18")  # True
        #   Decimal("3004.96") == Decimal("273.18")  # False
        # Therefore, only the match for 273,18 remains.
        # For total = "3004.96", only 3 004,96 remains.
        candidates = [
            candidate
            for candidate in available
            if normalize_number(
                candidate.group(),
                percentage=is_percentage,
            ) == expected_number
        ]

        # Require one unambiguous match
        # Skip the field when:
        # - There are zero numerically equal candidates.
        # - The same expected number occurs in multiple candidate positions.
        if len(candidates) != 1:
            continue
        
        # The function doesn’t return a review record itself
        # the missing annotation is detected by your verification code.
        # For tax:
        #   path = "subtotal.tax"
        #   observed = "273,18"
        candidate = candidates[0]
        path = f"subtotal.{field}"
        observed = candidate.group()

        # Build the annotation
        #   * start: original-text starting offset.
        #   * end: exclusive ending offset.
        #   * label: field name.
        #   * text: original OCR substring.
        #   * score: diagnostic text similarity.
        #   * match_status: records how the annotation was accepted.
        #   * needs_review=False: it passed this function’s uniqueness and numeric checks.
        result[path] = {
            "start": candidate.start(),
            "end": candidate.end(),
            "label": path,
            "text": observed,
            "score": fuzz.ratio(
                normalize(expected),
                normalize(observed),
            ),
            "match_status": "summary_numeric_equal",
            "needs_review": False,
        }

    return result
    

def find_spans(text, json_data, prefix=""):
    entities = []
    stack = [(json_data, "")]
    search_from = {"quantity": 0, "total_price": 0}
    norm_text, offsets = normalize_with_offsets(text)
    summary_spans = find_summary_spans(text, json_data)

    while stack:
        current, path = stack.pop()

        if isinstance(current, dict):
            for key, value in current.items():
                child_path = f"{path}.{key}" if path else key
                stack.append((value, child_path))
            continue

        if isinstance(current, list):
            for idx in range(len(current) - 1, -1, -1):
                stack.append((current[idx], f"{path}[{idx}]"))
            continue

        if not isinstance(current, str) or not current.strip():
            continue
        
        if "items" in path:
            continue
        if path in {"subtotal.tax", "subtotal.total"}:
            entity = summary_spans.get(path)
            if entity is not None:
                entities.append(entity)
            continue
        if path.endswith("_name"):
            entity = find_name_span(text, current, path)
            if entity is not None:
                entities.append(entity)
            continue

        if "account_number" in path:
            norm_value, _ = normalize_with_offsets(current)
            match = fuzz.partial_ratio_alignment(norm_value,norm_text)
            if not match:
                continue
            start = offsets[match.dest_start]
            end = offsets[match.dest_end-1] + 1
            score = fuzz.ratio(normalize(current, path), normalize(text[start:end], path))
            entities.append({
                "start": start,
                "end": end,
                "label": path,
                "text": text[start:end],
                "needs_review": score < ACCOUNT_NUMBER_MIN_SCORE,
                "score": score,
            })
            continue

        if is_numeric_field(path):
            pattern = number_pattern(current)
            cursor_key = path.rsplit(".", 1)[-1].split("[", 1)[0]
            start_at = search_from.get(cursor_key, 0)
            match = pattern.search(text, start_at)
            if not match:
                continue
            start, end = match.span()
            entities.append({
                "start": start,
                "end": end,
                "label": path,
                "text": text[start:end],
            })
            search_from[cursor_key] = end

            continue

        if path.endswith("_address"):
            norm_text, offsets = normalize_with_offsets(text)
            for line in current.splitlines():
                line = line.strip()
                if not line:
                    continue

                entity = find_normalized_span(text, line, path)
                if entity is not None:
                    entity["needs_review"] = False
                    entity["match_status"] = "exact"
                    entities.append(entity)
                    continue
                norm_line, _ = normalize_with_offsets(line)
                match = fuzz.partial_ratio_alignment(norm_line,norm_text)
                if not match or match.dest_end <= match.dest_start:
                    continue

                start = offsets[match.dest_start]
                end = offsets[match.dest_end-1] + 1
                house_number = re.match(r"\d+\b", line.strip())
                if house_number:
                    number = house_number.group()

                    # Search within the fuzzy candidate, preserving original offsets.
                    candidate_text = text[start:end]
                    anchor = re.search(rf"(?<!\w){re.escape(number)}(?!\w)", candidate_text)

                    if anchor:
                        start += anchor.start()

                matched_text = text[start:end]
                entities.append({
                    "start": start,
                    "end": end,
                    "label": path,
                    "text": matched_text,
                })

            continue

        norm_current, _ = normalize_with_offsets(current)
        match = fuzz.partial_ratio_alignment(norm_current,norm_text)
        if not match:
            continue
        start = offsets[match.dest_start]
        end = offsets[match.dest_end-1] + 1
        entities.append({
            "start": start,
            "end": end,
            "label": path,
            "text": text[start:end],
            "needs_review": (
                path.endswith("_name") or path.endswith("_number")
            ) and normalize(current, path) != normalize(text[start:end], path),
            "score": match.score,
        })

    entities.sort(key=lambda entity: entity["start"])

    return {
        "text": text,
        "entities": entities,
    }

def normalize_identifier(value: str) -> str:
    """Ignore identifier spacing and separators while preserving letters/digits."""
    value = unicodedata.normalize("NFKC", str(value)).casefold()
    return re.sub(r"[\s./\-]", "", value)


def normalize(value: str, path: str = "") -> str:
    """Normalize text, or an identifier when its field path is supplied."""
    if path.rsplit(".", 1)[-1] in {"invoice_number", "account_number"}:
        return normalize_identifier(value)
    normalized, _ = normalize_with_offsets(str(value))
    return normalized

def flatten_field_paths(
    data: dict,
    prefix: str = "",
) -> list[dict]:
    """Return dotted paths for non-empty string values in a nested JSON object."""
    paths = []

    for key, value in data.items():
        path = f"{prefix}.{key}" if prefix else key

        if isinstance(value, dict):
            paths.extend(flatten_field_paths(value, path))
        elif isinstance(value, list):
            for idx, item in enumerate(value):
                if isinstance(item, dict):
                    paths.extend(flatten_field_paths(item, f"{path}[{idx}]"))
        elif isinstance(value, str) and value.strip():
            paths.append({path: value})

    return paths

def iter_leaf_fields(data, path=""):
    """Yield (field_path, value), keeping list items under the same path."""
    if isinstance(data, dict):
        for key, value in data.items():
            child_path = f"{path}.{key}" if path else key
            yield from iter_leaf_fields(value, child_path)

    elif isinstance(data, list):
        for item in data:
            yield from iter_leaf_fields(item, path)

    elif data is not None and str(data).strip():
        yield path, data

def normalize_number(value: str, *, percentage: bool = False) -> Decimal | None:
    """Parse invoice amounts with up to two decimal places and valid grouping.

    Accept comma/dot decimals, grouped thousands, and surrounding currency
    symbols. Reject ambiguous single separators (1,000), malformed grouping,
    arbitrary letters, and multiple amounts instead of silently merging them.
    With percentage=True, require a trailing percent sign and return percentage
    points (10% becomes Decimal('10'), not Decimal('0.1')).
    """
    value = unicodedata.normalize("NFKC", str(value)).strip()
    if percentage:
        if not value.endswith("%"):
            return None
        value = value[:-1].strip()
    value = value.strip("$€£¥ ").strip()
    value = value.replace("’", "'")
    sign = ""
    if value.startswith(("+", "-")):
        sign, value = value[0], value[1:]

    if not value or not re.fullmatch(r"[0-9.,'\s]+", value):
        return None

    integer_part = value
    fractional_part = ""
    separators = {char for char in value if char in ".,"}
    if separators:
        decimal_separator = value[max(value.rfind("."), value.rfind(","))]
        before, after = value.rsplit(decimal_separator, 1)
        if re.fullmatch(r"[0-9]{1,2}", after):
            integer_part, fractional_part = before, after
            if decimal_separator in integer_part:
                return None
        elif len(separators) == 1 and value.count(decimal_separator) >= 2:
            # Repeated three-digit groups have an unambiguous integer reading.
            pass
        else:
            return None

    if not re.fullmatch(r"[0-9]+", integer_part):
        if not re.fullmatch(r"[0-9]{1,3}([.,'\s])[0-9]{3}(?:\1[0-9]{3})*", integer_part):
            return None
    integer_digits = re.sub(r"[^0-9]", "", integer_part)
    canonical = sign + integer_digits
    if fractional_part:
        canonical += "." + fractional_part

    try:
        return Decimal(canonical)
    except InvalidOperation:
        return None

def find_fuzzy_identifier(text: str, expected: str, min_score: float = 90):
    expected = expected.strip()
    if not expected:
        return None

    # Try same-length OCR windows: identifiers should not gain or lose characters.
    width = len(expected)
    candidates = []

    for start in range(len(text) - width + 1):
        candidate = text[start : start + width]
        score = fuzz.ratio(expected, candidate)

        if score >= min_score:
            candidates.append((score, start, start + width, candidate))

    # Return only an unambiguous best candidate.
    candidates.sort(reverse=True)
    if not candidates:
        return None

    best = candidates[0]
    if len(candidates) > 1 and candidates[1][0] == best[0]:
        return None

    score, start, end, candidate = best
    return {
        "start": start,
        "end": end,
        "text": candidate,       # OCR text, unchanged
        "score": score,
        "expected": expected,    # keep expected JSON separately
    }


def find_unmatched_fields(row):
    """Log and return missing, mismatched, and review-required base fields."""
    ocr_text = row["ocred_text"]
    json_data = row["json_data"]

    if isinstance(json_data, str):
        json_data = json.loads(json_data)

    result_dict, expected_dict = {}, {}
    result = find_spans(ocr_text, json_data)
    json_fields = flatten_field_paths(json_data)

    for item in result['entities']:
        if item['label'] in result_dict:
            result_dict[item['label']] += f"\n{item['text']}"
        else:
            result_dict[item['label']] = item['text']

    for item in json_fields:
        for key, value in item.items():
            if key in expected_dict:
                expected_dict[key] += f"\n{value}"
            else:
                expected_dict[key] = value

    issues = []

    for key, value in expected_dict.items():
        orig_value = result_dict.get(key)

        if "items" in key:
            continue

        if not orig_value:
            issues.append({"field": key, "status": "missing", "expected": value,
                           "observed": None})
            continue
        
        if is_numeric_field(key):
            is_percentage = unicodedata.normalize("NFKC", value).strip().endswith("%")
            expected_number = normalize_number(value, percentage=is_percentage)
            observed_number = normalize_number(orig_value, percentage=is_percentage)

            score = fuzz.ratio(str(expected_number), str(observed_number))
            is_match = (
                expected_number is not None
                and observed_number is not None
                and expected_number == observed_number
            )

            if not is_match: #score < 80:
                issues.append({"field": key, "status": "numeric_mismatch", "expected": value,
                               "observed": orig_value})
            continue

        if key.endswith(("_name", "_number")):
            score = fuzz.ratio(normalize(value, key), normalize(orig_value, key))
            field_spans = [entity for entity in result["entities"] if entity["label"] == key]
            if key.endswith(".account_number"):
                requires_review = score < ACCOUNT_NUMBER_MIN_SCORE
            elif key.endswith("_name"):
                requires_review = any(entity.get("needs_review", True) for entity in field_spans)
            else:
                requires_review = normalize(value, key) != normalize(orig_value, key)
            if requires_review:
                issues.append({"field": key, "status": "needs_review", "expected": value,
                               "observed": orig_value, "score": score,
                               "spans": field_spans})
            continue

        # A subset of address words must not count as a complete match.
        ratio = fuzz.ratio(normalize(value, key), normalize(orig_value, key))

        if ratio < 80:
            issues.append({"field": key, "status": "text_mismatch", "expected": value,
                           "observed": orig_value, "score": ratio})
            continue

    return issues


def find_ignored_account_regions(text, json_data):
    account = (
        json_data.get("payment_instructions", {})
        .get("account_number")
    )

    if account is not None and str(account).strip():
        return []

    # Assumes this dataset's IBAN value is one whitespace-delimited token.
    pattern = re.compile(
        r"\bIBAN\s*:\s*(?P<value>[A-Za-z0-9][A-Za-z0-9/.-]*)",
        flags=re.IGNORECASE,
    )

    return [
        {
            "start": match.start("value"),
            "end": match.end("value"),
        }
        for match in pattern.finditer(text)
    ]

def prepare_records(frame):
    FIELD_PATHS = [
        "invoice.client_name",
        "invoice.seller_name",
        "invoice.client_address",
        "invoice.seller_address",
        "invoice.invoice_number",
        "invoice.invoice_date",
        "payment_instructions.account_number",
        "subtotal.tax",
        "subtotal.total",
    ]
    FIELD_PATHS_TO_LABEL = {
        "invoice.client_name"
    }

    LABELS = ["O"]

    for field in FIELD_PATHS:
        LABELS.extend([f"B-{field}", f"I-{field}"])

    label2id = {label: index for index, label in enumerate(LABELS)}
    id2label = dict(enumerate(LABELS))

    records = []
    rejected = []

    for row_index, row in frame.iterrows():
        target = row["json_data"]

        if isinstance(target, str):
            target = json.loads(target)

        text = row["ocred_text"]

        # Keep parser diagnostics without flooding notebook output.
        with contextlib.redirect_stdout(io.StringIO()) as log:
            issues = find_unmatched_fields(row)

        if not isinstance(issues, list):
            raise TypeError(
                "find_unmatched_fields() must return a list."
            )

        issues = [
            issue
            for issue in issues
            if issue["field"] in FIELD_PATHS
        ]

        result = find_spans(text, target)

        entities = [
            entity
            for entity in result["entities"]
            if entity["label"] in FIELD_PATHS
        ]

        entities.sort(
            key=lambda entity: (entity["start"], entity["end"])
        )

        for entity in entities:
            start, end = entity["start"], entity["end"]

            if not (
                0 <= start < end <= len(text)
                and text[start:end] == entity["text"]
            ):
                raise ValueError(
                    f"Invalid span in invoice {row_index}: {entity}"
                )

            if entity.get("needs_review", False):
                issues.append({
                    "field": entity["label"],
                    "status": "needs_review",
                })

        for left, right in zip(entities, entities[1:]):
            if right["start"] < left["end"]:
                issues.append({
                    "field": right["label"],
                    "status": "overlap",
                })

        # The parser's similarity threshold can accept partial addresses.
        # Require complete normalized address coverage for this baseline.
        for field in (
            "invoice.client_address",
            "invoice.seller_address",
        ):
            key = field.split(".")[1]
            expected = target.get("invoice", {}).get(key, "")

            observed = " ".join(
                entity["text"]
                for entity in entities
                if entity["label"] == field
            )

            if normalize(expected) != normalize(observed):
                issues.append({
                    "field": field,
                    "status": "partial_address",
                })

        if issues:
            rejected.append({
                "row_index": row_index,
                "issues": issues,
                "log": log.getvalue(),
            })
        else:
            records.append({
                "row_index": row_index,
                "text": text,
                "entities": entities,
                "ignored_regions": find_ignored_account_regions(text, target)
            })

    return records, rejected


In [19]:
find_spans(ocr_text, json_data)


{'text': 'Invoice no: 84652373 Date of issue: 02/23/2021 Seller: Client: Nguyen-Roach Clark-Foster 247 David Highway 77477 Cliff Apt. 853 Lake John, WV 84178 Washingtonbury, MS 78346 Tax Id: 991-72-5826 Tax Id: 937-70-8530 IBAN: GB91/YXO05542456978150 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Stemware Rack Display Kitchen 1,00 each 42,32 42,32 10% 46,55 Wine Glass Holder Bottle Carbon Steel Free Punch 2 VTG (4) 7 Ounce Since 1852 1,00 each 14,00 14,00 10% 15,40 Milk Bottle Wine Carafe Juice Glass with Cork Lids 3 Vintage Crystal Red Wine 1,00 each 35,45 35,45 10% 39,00 Glasses NOS West Germany 1983 6 10 ounce elegant stems 3 Ikea Stainless Steel 4-bottle 4,00 each 25,00 100,00 10% 110,00 Wine Rack 300.557.60 great condition gift it! 5 Lolita "Wine Bouquet" Hand 1,00 each 20,00 20,00 10% 22,00 Painted and Decorated Wine Glass NIB SUMMARY VAT [%] Net worth VAT Gross worth 10% 211,77 21,18 232,95 Total $ 211,77 $ 21,18 $ 232,95 Troy',
 'entities': [{'start': 12,

In [53]:
!ls -la /content/drive/MyDrive/ocr

total 2141
drwx------  6 root root    4096 Oct  6 09:42 .
drwx------ 14 root root    4096 Oct 29 09:25 ..
-rw-------  1 root root 2167230 Oct  6 09:43 annotated_invoices.parquet
drwx------  2 root root    4096 Oct  6 11:20 bert
drwx------  2 root root    4096 Oct 18 11:42 bert-invoice-baseline
drwx------  2 root root    4096 Oct 18 13:55 bert-invoice-v2
drwx------  2 root root    4096 Oct  6 16:27 bert-tax-total-v2


In [111]:
duplicate_key = df["ocred_text"].str.replace(
    r"\s+", " ", regex=True
)
df = df.loc[~duplicate_key.duplicated()]

records, rejected = prepare_records(df)

print("Accepted:", len(records))
print("Excluded:", len(rejected))

Accepted: 1306
Excluded: 107


In [21]:
pd.DataFrame(rejected).head()

,row_index,issues,log
0,0,"[{'field': 'invoice.client_address', 'status':...",
1,3,"[{'field': 'invoice.seller_address', 'status':...",
2,10,"[{'field': 'invoice.client_address', 'status':...",
3,13,"[{'field': 'invoice.client_address', 'status':...",
4,31,"[{'field': 'invoice.client_address', 'status':...",


## Dataset and Dataloader

Split invoices before creating token windows. For a controlled comparison with previous training, reuse its exact document splits instead of generating new ones.

In [ ]:
shuffled = records.copy()
random.Random(SEED).shuffle(shuffled)

train_records, remaining_records = train_test_split(
    records,
    test_size=0.2,
    shuffle=True,
    random_state=SEED,
)

validation_records, test_records = train_test_split(
    remaining_records,
    test_size=0.5,
    shuffle=True,
    random_state=SEED,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

In [113]:
class InvoiceDataset(Dataset):
    """Convert OCR spans into BIO labels for consecutive token windows."""

    def __init__(self, records, tokenizer, label2id, max_length=512):
        self.examples = []

        for record in records:
            encoded = tokenizer(
                record["text"],
                truncation=True,
                max_length=max_length,
                return_offsets_mapping=True,
                return_overflowing_tokens=True,
                stride=0,
            )

            for window, offsets in enumerate(
                encoded["offset_mapping"]
            ):
                labels = []
                previous_entity = None

                for start, end in offsets:
                    if start == end:
                        labels.append(-100)
                        previous_entity = None
                        continue

                    overlaps_ignored_region = any(
                        start < region["end"] and end > region["start"]
                        for region in record.get("ignored_regions", [])
                    )

                    if overlaps_ignored_region:
                        labels.append(-100)
                        previous_entity = None
                        continue

                    overlapping = [
                        (index, entity)
                        for index, entity in enumerate(
                            record["entities"]
                        )
                        if start < entity["end"]
                        and end > entity["start"]
                    ]

                    if not overlapping:
                        labels.append(label2id["O"])
                        previous_entity = None
                        continue

                    if len(overlapping) != 1:
                        labels.append(-100)
                        previous_entity = None
                        continue

                    entity_index, entity = overlapping[0]

                    # Ignore tokens that cross annotation boundaries.
                    if (
                        start < entity["start"]
                        or end > entity["end"]
                    ):
                        labels.append(-100)
                        previous_entity = None
                        continue

                    prefix = (
                        "I"
                        if previous_entity == entity_index
                        else "B"
                    )

                    entity_type = FIELD_TO_ENTITY[entity["label"]]
                    bio_label = f"{prefix}-{entity_type}"

                    labels.append(label2id[bio_label])
                    previous_entity = entity_index

                example = {
                    key: encoded[key][window]
                    for key in tokenizer.model_input_names
                    if key in encoded
                }
                example["labels"] = labels

                self.examples.append(example)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

In [114]:
label2id

{'O': 0,
 'B-CLIENT_ADDRESS': 1,
 'I-CLIENT_ADDRESS': 2,
 'B-CLIENT_NAME': 3,
 'I-CLIENT_NAME': 4,
 'B-SELLER_ADDRESS': 5,
 'I-SELLER_ADDRESS': 6,
 'B-SELLER_NAME': 7,
 'I-SELLER_NAME': 8,
 'B-INVOICE_NUMBER': 9,
 'I-INVOICE_NUMBER': 10,
 'B-INVOICE_DATE': 11,
 'I-INVOICE_DATE': 12,
 'B-ACCOUNT_NUMBER': 13,
 'I-ACCOUNT_NUMBER': 14,
 'B-TAX': 15,
 'I-TAX': 16,
 'B-TOTAL': 17,
 'I-TOTAL': 18}

In [115]:
train_dataset = InvoiceDataset(
    train_records, tokenizer, label2id, MAX_LENGTH
)
validation_dataset = InvoiceDataset(
    validation_records, tokenizer, label2id, MAX_LENGTH
)
test_dataset = InvoiceDataset(
    test_records, tokenizer, label2id, MAX_LENGTH
)

In [116]:
collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collator,
)

### Inspect batch

In [117]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)

torch.Size([16, 376])
torch.Size([16, 376])


In [118]:
print("Training windows:", len(train_dataset))
print("Updates per epoch:", len(train_loader))
print("Updates over five epochs:", 5 * len(train_loader))

Training windows: 1044
Updates per epoch: 66
Updates over five epochs: 330


In [119]:
tokens = tokenizer.convert_ids_to_tokens(
    batch["input_ids"][0]
)
" ".join(tokens[:30])

'[CLS] in ##vo ##ice no : 1729 ##0 ##43 ##0 date of issue : 05 / 27 / 2013 seller : client : hodge - stokes hamilton ltd 405 christensen'

In [120]:
for token, label_id in zip(tokens, batch["labels"][0].tolist()):
    if label_id != -100:
        print(token, id2label[label_id])

in O
##vo O
##ice O
no O
: O
1729 B-INVOICE_NUMBER
##0 I-INVOICE_NUMBER
##43 I-INVOICE_NUMBER
##0 I-INVOICE_NUMBER
date O
of O
issue O
: O
05 B-INVOICE_DATE
/ I-INVOICE_DATE
27 I-INVOICE_DATE
/ I-INVOICE_DATE
2013 I-INVOICE_DATE
seller O
: O
client O
: O
hodge B-SELLER_NAME
- I-SELLER_NAME
stokes I-SELLER_NAME
hamilton B-CLIENT_NAME
ltd I-CLIENT_NAME
405 B-SELLER_ADDRESS
christensen I-SELLER_ADDRESS
loop I-SELLER_ADDRESS
98 B-CLIENT_ADDRESS
##9 I-CLIENT_ADDRESS
##9 I-CLIENT_ADDRESS
bryan I-CLIENT_ADDRESS
springs I-CLIENT_ADDRESS
brown B-SELLER_ADDRESS
##bury I-SELLER_ADDRESS
, I-SELLER_ADDRESS
oh I-SELLER_ADDRESS
52 I-SELLER_ADDRESS
##22 I-SELLER_ADDRESS
##1 I-SELLER_ADDRESS
clark B-CLIENT_ADDRESS
##shire I-CLIENT_ADDRESS
, I-CLIENT_ADDRESS
ar I-CLIENT_ADDRESS
57 I-CLIENT_ADDRESS
##18 I-CLIENT_ADDRESS
##4 I-CLIENT_ADDRESS
tax O
id O
: O
94 O
##3 O
- O
98 O
- O
83 O
##53 O
tax O
id O
: O
99 O
##6 O
- O
90 O
- O
61 O
##8 O
##7 O
ib O
##an O
: O
items O
no O
. O
description O
q O
##ty O
u

## Pytorch model

This fine-tunes pretrained BERT with a dropout layer and linear classifier.

In [121]:
class BertFieldExtractor(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name
        )
        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels,
        )

    def forward(self, input_ids, attention_mask):
        hid = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        return self.classifier(hid)

### Test

In [52]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

test_model = BertFieldExtractor(
    model_name=MODEL_NAME,
    num_labels=len(LABELS)
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [122]:
logits = test_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
)

loss = criterion(
    logits.reshape(-1, logits.shape[-1]),
    batch["labels"].reshape(-1),
)

## Train functions

In [79]:
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
)
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from tqdm.notebook import tqdm

### Train one epoch

In [123]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    progress_bar = tqdm(loader, position=1, desc=f"Train epoch {epoch}", leave=False)
    for batch in progress_bar:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            batch["labels"].reshape(-1),
        )

        loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )
        optimizer.step()

        # Weight the epoch loss by the number of supervised tokens.
        valid_tokens = (batch["labels"] != -100).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    if total_tokens == 0:
        raise ValueError("Training loader contains no supervised tokens.")

    return {"loss": total_loss / total_tokens}

### Eval one epoch

In [124]:
@torch.inference_mode()
def eval_one_epoch(model, loader, criterion, device, id2label, epoch):
    model.eval()

    num_labels = len(id2label)

    accuracy = MulticlassAccuracy(
        num_classes=num_labels,
        average="micro",
    ).to(device)

    macro_f1 = MulticlassF1Score(
        num_classes=num_labels,
        average="macro",
    ).to(device)

    actual_sequences = []
    predicted_sequences = []

    total_loss = 0.0
    total_tokens = 0

    progress_bar = tqdm(loader, position=1, desc=f"Eval epoch {epoch}", leave=False)
    for batch in progress_bar:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }
        targets = batch["labels"]

        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            targets.reshape(-1),
        )

        predictions = logits.argmax(dim=-1)

        valid = targets != -100
        valid_tokens = valid.sum().item()

        accuracy.update(predictions[valid], targets[valid])
        macro_f1.update(predictions[valid], targets[valid])

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

        for predicted_row, target_row in zip(
            predictions.cpu().tolist(),
            targets.cpu().tolist(),
        ):
            actual_sequences.append([
                id2label[target]
                for target in target_row
                if target != -100
            ])

            predicted_sequences.append([
                id2label[predicted]
                for predicted, target in zip(predicted_row, target_row)
                if target != -100
            ])

    if total_tokens == 0:
        raise ValueError("Evaluation loader contains no supervised tokens.")

    return {
        "loss": total_loss / total_tokens,
        "precision": precision_score(
            actual_sequences,
            predicted_sequences,
            zero_division=0,
        ),
        "recall": recall_score(
            actual_sequences,
            predicted_sequences,
            zero_division=0,
        ),
        "f1": f1_score(
            actual_sequences,
            predicted_sequences,
            zero_division=0,
        ),
        "token_accuracy": accuracy.compute().item(),
        "token_macro_f1": macro_f1.compute().item(),
        "per_entity": classification_report(
            actual_sequences,
            predicted_sequences,
            output_dict=True,
            zero_division=0,
        ),
    }


### Train

In [125]:
def train(
    model,
    train_loader,
    validation_loader,
    optimizer,
    criterion,
    device,
    id2label,
    epochs=5,
):
    if epochs < 1:
        raise ValueError("epochs must be positive.")

    model.to(device)

    best_f1 = -1.0
    best_state = None
    history = []

    progress_bar = tqdm(range(1, epochs + 1), position=0, leave=True)
    for epoch in progress_bar:
        progress_bar.set_description(f"Train model {epoch}/{epochs+1}")
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            epoch=epoch
        )

        validation_metrics = eval_one_epoch(
            model=model,
            loader=validation_loader,
            criterion=criterion,
            device=device,
            id2label=id2label,
            epoch=epoch
        )

        epoch_metrics = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            **{
                f"validation_{key}": value
                for key, value in validation_metrics.items()
                if key != "per_entity"
            },
        }

        history.append(epoch_metrics)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"train loss: {train_metrics['loss']:.4f} | "
            f"validation loss: {validation_metrics['loss']:.4f} | "
            f"validation F1: {validation_metrics['f1']:.4f}"
        )

        if validation_metrics["f1"] > best_f1:
            best_f1 = validation_metrics["f1"]

            # Copy the tensors; state_dict() alone shares model storage.
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)

    return history_df

In [126]:
model = BertFieldExtractor(
    model_name=MODEL_NAME,
    num_labels=len(LABELS)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
)

criterion = nn.CrossEntropyLoss(ignore_index=-100)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [127]:
history_df = train(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    id2label=id2label,
    epochs=5,
)
history_df.to_csv('bert_training_colab.csv')

  0%|          | 0/5 [00:00<?, ?it/s]

Train epoch 1:   0%|          | 0/66 [00:00<?, ?it/s]

Eval epoch 1:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 1/5 | train loss: 0.5447 | validation loss: 0.1036 | validation F1: 0.6309


Train epoch 2:   0%|          | 0/66 [00:00<?, ?it/s]

Eval epoch 2:   0%|          | 0/9 [00:00<?, ?it/s]

... (6 outputs left out)

Epoch 4/5 | train loss: 0.0082 | validation loss: 0.0055 | validation F1: 0.9955


Train epoch 5:   0%|          | 0/66 [00:00<?, ?it/s]

Eval epoch 5:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 5/5 | train loss: 0.0067 | validation loss: 0.0045 | validation F1: 0.9970


Use these metrics together:

| Metric | Purpose |
|---|---|
| Entity precision | How many predicted entities are correct |
| Entity recall | How many annotated entities were recovered |
| Entity F1 | Main checkpoint-selection metric |
| Token accuracy | General token-label correctness, including O |
| Token macro F1 | Performance across BIO label classes |
| Per-entity F1 | Which fields need improvement |

In [132]:
history_df

,epoch,train_loss,validation_loss,validation_precision,validation_recall,validation_f1,validation_token_accuracy,validation_token_macro_f1
0,1,0.544703,0.103637,0.613813,0.649031,0.630931,0.970727,0.715218
1,2,0.053025,0.011474,0.974132,0.982116,0.978108,0.998535,0.994256
2,3,0.012220,0.006830,0.994039,0.994039,0.994039,0.999062,0.996963
3,4,0.008235,0.005455,0.995529,0.995529,0.995529,0.999238,0.997373
4,5,0.006650,0.004498,0.997019,0.997019,0.997019,0.999326,0.997581


In [133]:
!cat bert_training_colab.csv

,epoch,train_loss,validation_loss,validation_precision,validation_recall,validation_f1,validation_token_accuracy,validation_token_macro_f1
0,1,0.5447032361681817,0.10363719302892482,0.6138125440451022,0.6490312965722802,0.6309308221658819,0.9707269668579102,0.7152184247970581
1,2,0.053025269877314395,0.011473979811541688,0.9741315594974131,0.9821162444113264,0.9781076066790353,0.9985348582267761,0.9942563772201538
2,3,0.012219539705196455,0.006829785695924445,0.9940387481371088,0.9940387481371088,0.9940387481371088,0.9990622997283936,0.9969631433486938
3,4,0.00823497345924866,0.005454912120668623,0.9955290611028316,0.9955290611028316,0.9955290611028316,0.999238133430481,0.9973734617233276
4,5,0.006650470727560555,0.004497858237853368,0.9970193740685543,0.9970193740685543,0.9970193740685543,0.9993260502815247,0.9975810050964355


## Inspect results

In [134]:
test_metrics = eval_one_epoch(
    model,
    test_loader,
    criterion=criterion,
    device=device,
    id2label=id2label,
    epoch=1
)

Eval epoch 1:   0%|          | 0/9 [00:00<?, ?it/s]

In [136]:
entity_report = pd.DataFrame(test_metrics["per_entity"]).T
entity_report


,precision,recall,f1-score,support
ACCOUNT_NUMBER,1.000000,1.000000,1.000000,30.0
CLIENT_ADDRESS,1.000000,1.000000,1.000000,262.0
CLIENT_NAME,1.000000,1.000000,1.000000,131.0
INVOICE_DATE,1.000000,1.000000,1.000000,131.0
INVOICE_NUMBER,1.000000,1.000000,1.000000,131.0
SELLER_ADDRESS,1.000000,1.000000,1.000000,262.0
SELLER_NAME,1.000000,1.000000,1.000000,131.0
TAX,1.000000,1.000000,1.000000,131.0
TOTAL,0.984733,0.984733,0.984733,131.0
micro avg,0.998507,0.998507,0.998507,1340.0


## Simple inference: OCR text => spans => nested JSON

Inference uses the trained model directly. It does not call find_spans() or access the original JSON.

First, predict BIO labels and reconstruct spans. This handles long documents with consecutive, nonoverlapping windows:

In [150]:
@torch.inference_mode()
def predict_spans(
    text,
    model,
    tokenizer,
    device,
    id2label,
    max_length=512,
):
    model.eval()

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
        stride=0,
    )

    entities = []
    active = None

    for window_index, offsets in enumerate(encoded["offset_mapping"]):
        inputs = {
            key: torch.tensor(
                [encoded[key][window_index]],
                device=device,
            )
            for key in tokenizer.model_input_names
            if key in encoded
        }

        logits = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
        probabilities = logits.softmax(dim=-1)[0]
        confidences, predictions = probabilities.max(dim=-1)

        for (start, end), predicted_id, confidence in zip(
            offsets,
            predictions.cpu().tolist(),
            confidences.cpu().tolist(),
        ):
            # Skip special tokens.
            if start == end:
                continue

            tag = id2label[predicted_id]

            if tag == "O":
                active = None
                continue

            prefix, field = tag.split("-", 1)

            # An orphan I label starts a new entity.
            if (
                prefix == "I"
                and active is not None
                and active["label"] == field
            ):
                active["end"] = end
                active["scores"].append(confidence)
            else:
                active = {
                    "start": start,
                    "end": end,
                    "label": field,
                    "scores": [confidence],
                }
                entities.append(active)

    for entity in entities:
        scores = entity.pop("scores")

        entity["text"] = text[entity["start"]:entity["end"]]
        entity["confidence"] = sum(scores) / len(scores)

    return entities

In [139]:
FIELD_PATHS = [
    "invoice.client_name",
    "invoice.seller_name",
    "invoice.client_address",
    "invoice.seller_address",
    "invoice.invoice_number",
    "invoice.invoice_date",
    "payment_instructions.account_number",
    "subtotal.tax",
    "subtotal.total",
]

In [140]:
def spans_to_json(entities):
    result = {}

    for field in FIELD_PATHS:
        group, key = field.split(".")
        result.setdefault(group, {})[key] = None

        candidates = [
            entity
            for entity in entities
            if entity["label"] == field
        ]

        if not candidates:
            continue

        if field.endswith("_address"):
            candidates.sort(key=lambda entity: entity["start"])

            # Preserve separate address fragments in reading order.
            fragments = list(dict.fromkeys(
                candidate["text"]
                for candidate in candidates
            ))
            result[group][key] = "\n".join(fragments)
        else:
            # Simple policy for repeated scalar predictions.
            best = max(
                candidates,
                key=lambda candidate: candidate["confidence"],
            )
            result[group][key] = best["text"]

    return result

In [142]:
ocr_text = test_records[0]["text"]
ocr_text

"Invoice no: 77006579 Date of issue: 11/12/2019 Seller: Client: Wilson Inc Hunter-Miranda 410 Jesus Street Unit 2895 Box 4028 Christopherville, SD 23249 DPO AP 06411 Tax Id: 928-92-2371 Tax Id: 935-99-6545 IBAN: GB3BLDMS35426816092509 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth 1. Divided Black Asymmetrical 4,00 each 7,99 31,96 10% 35,16 Dress -size 8 NWT 2 Vintage 90s Sostanza Plus Size 2,00 each 49,96 99,92 10% 109,91 Dress 22W Floral Cottagecore 3 LuLaRoe Nicole Dress Size Large 3,00 each 1,99 5,97 10% 6,57 26 Pre-Owned Women's Nah Nah 2,00 each 12,99 25,98 10% 28,58 Dress Size 8 Black Sheer Formal 5 GUESS dress size 2 5,00 each 20,00 100,00 10% 110,00 LuLaRoe Nicole Dress Size 1,00 each 1,99 1,99 10% 2,19 Medium 12 SUMMARY VAT [%] Net worth VAT Gross worth 10% 265,82 26,58 292,40 Total $ 265,82 $ 26,58 $ 292,40 Sexy"

In [152]:
predicted_spans = predict_spans(
    text=ocr_text,
    model=model,
    tokenizer=tokenizer,
    device=device,
    id2label=id2label,
)
predicted_spans

[{'start': 12,
  'end': 20,
  'label': 'INVOICE_NUMBER',
  'text': '77006579',
  'confidence': 0.9966148734092712},
 {'start': 36,
  'end': 46,
  'label': 'INVOICE_DATE',
  'text': '11/12/2019',
  'confidence': 0.9970546126365661},
 {'start': 63,
  'end': 73,
  'label': 'SELLER_NAME',
  'text': 'Wilson Inc',
  'confidence': 0.9952077269554138},
 {'start': 74,
  'end': 88,
  'label': 'CLIENT_NAME',
  'text': 'Hunter-Miranda',
  'confidence': 0.9953388770421346},
 {'start': 89,
  'end': 105,
  'label': 'SELLER_ADDRESS',
  'text': '410 Jesus Street',
  'confidence': 0.9969930251439413},
 {'start': 106,
  'end': 124,
  'label': 'CLIENT_ADDRESS',
  'text': 'Unit 2895 Box 4028',
  'confidence': 0.9972739020983378},
 {'start': 125,
  'end': 151,
  'label': 'SELLER_ADDRESS',
  'text': 'Christopherville, SD 23249',
  'confidence': 0.9979328016440073},
 {'start': 152,
  'end': 164,
  'label': 'CLIENT_ADDRESS',
  'text': 'DPO AP 06411',
  'confidence': 0.9976071119308472},
 {'start': 211,
  'end'

In [156]:
mapped_spans = [
    {
        **span,
        "label": ENTITY_TO_FIELD[span["label"]],
    }
    for span in predicted_spans
]


prediction = spans_to_json(mapped_spans)
prediction

{'invoice': {'client_name': 'Hunter-Miranda',
  'seller_name': 'Wilson Inc',
  'client_address': 'Unit 2895 Box 4028\nDPO AP 06411',
  'seller_address': '410 Jesus Street\nChristopherville, SD 23249',
  'invoice_number': '77006579',
  'invoice_date': '11/12/2019'},
 'payment_instructions': {'account_number': 'GB3BLDMS35426816092509'},
 'subtotal': {'tax': '26,58', 'total': '292,40'}}